In [3]:
# ── CONFIGURACION ──────────────────────────────────────────────────────────────
# Edita estas dos variables antes de ejecutar el notebook

ANOMES_CARPETA = "202512"   # Carpeta de entrada (ej. "202512"). None = la más reciente
ANOMES_OUTPUT  = None        # AñoMes del archivo de salida. None = igual que la carpeta
# ───────────────────────────────────────────────────────────────────────────────

In [4]:
import re
from pathlib import Path
import pandas as pd

BASE_DIR         = Path(r"C:\Users\KOT14\Documents\Integral\Insumos\Contabilidad")
SHEET_CANDIDATES = ["2025", "KOTS-GT"]


def get_folder(anomes_carpeta):
    if anomes_carpeta:
        folder = BASE_DIR / anomes_carpeta
        if not folder.exists():
            raise FileNotFoundError(f"No existe la carpeta: {folder}")
        return folder
    folders = sorted([f for f in BASE_DIR.iterdir() if f.is_dir() and re.fullmatch(r"\d{6}", f.name)])
    if not folders:
        raise FileNotFoundError(f"No se encontraron carpetas YYYYMM en {BASE_DIR}")
    return folders[-1]


def extract_tab_name(filepath):
    stem = filepath.stem
    match = re.search(r"\bAP\s+(\w+)\s+vf$", stem, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    match = re.search(r"(\w+)\s+vf$", stem, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return stem


def get_engine(filepath):
    return "xlrd" if filepath.suffix.lower() == ".xls" else "openpyxl"


def read_sheet(filepath):
    xls = pd.ExcelFile(filepath, engine=get_engine(filepath))
    for candidate in SHEET_CANDIDATES:
        if candidate in xls.sheet_names:
            return pd.read_excel(xls, sheet_name=candidate, header=0)
    first = xls.sheet_names[0]
    print(f"  [aviso] Hoja {SHEET_CANDIDATES} no encontrada en '{filepath.name}', usando '{first}'")
    return pd.read_excel(xls, sheet_name=first, header=0)


print("Funciones cargadas.")

Funciones cargadas.


In [5]:
folder = get_folder(ANOMES_CARPETA)
anomes = ANOMES_OUTPUT if ANOMES_OUTPUT else folder.name

print(f"Carpeta de entrada : {folder}")
print(f"AnoMes del output  : {anomes}")

excel_files = [
    f for f in folder.iterdir()
    if f.suffix.lower() in (".xlsx", ".xls", ".xlsm")
    and not f.name.startswith("~$")
    and re.search(r"\bAP\s+\w+\s+vf", f.stem, re.IGNORECASE)
]

if not excel_files:
    raise FileNotFoundError(f"No se encontraron archivos 'AP * vf' en {folder}")

print(f"Archivos encontrados: {[f.name for f in excel_files]}\n")

output_path = folder / f"Payments_{anomes}.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for filepath in sorted(excel_files):
        tab_name = extract_tab_name(filepath)
        print(f"  Leyendo '{filepath.name}' -> pestana '{tab_name}' ...", end=" ")
        df = read_sheet(filepath)
        df.to_excel(writer, sheet_name=tab_name, index=False)
        print(f"({len(df)} filas)")

print(f"\nArchivo generado : {output_path}")

Carpeta de entrada : C:\Users\KOT14\Documents\Integral\Insumos\Contabilidad\202512
AnoMes del output  : 202512
Archivos encontrados: ['260106 2025 12 AP AE vf.xlsx', '260106 2025 12 AP CL vf.xlsx', '260106 2025 12 AP VA vf.xls']

  Leyendo '260106 2025 12 AP AE vf.xlsx' -> pestana 'AE' ... (184 filas)
  Leyendo '260106 2025 12 AP CL vf.xlsx' -> pestana 'CL' ... (320 filas)
  Leyendo '260106 2025 12 AP VA vf.xls' -> pestana 'VA' ... (119 filas)

Archivo generado : C:\Users\KOT14\Documents\Integral\Insumos\Contabilidad\202512\Payments_202512.xlsx
